# Fase 1 · extracción
Modelo doble determinista. LCEL, @chain, invoke, batch y stream.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from langchain_core.runnables import RunnableLambda
from pedro.chains import crear_lcel, crear_chain
import json
modelo = RunnableLambda(lambda prompt: json.dumps({'intencion':'comparar',
    'solicitud':{'producto_id':'PEC-01-0001','cantidad':2,'presupuesto_centavos':100000,
    'plazo_max_dias':7},'faltantes':[]}, ensure_ascii=False))
lcel, decorada = crear_lcel(modelo), crear_chain(modelo)
entrada = {'texto':'Compara dos unidades de PEC-01-0001 hasta R$ 1.000 en 7 días'}
print(lcel.invoke(entrada).model_dump_json())
print(decorada.invoke(entrada).model_dump_json())
print([x.model_dump() for x in lcel.batch([entrada, entrada])])
print(list(modelo.stream('etapa del modelo')))
print(lcel.get_graph().draw_mermaid())

{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null},"faltantes":[]}
{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null},"faltantes":[]}
[{'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None}, 'faltantes': []}, {'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None}, 'faltantes': []}]
['{"intencion": "comparar", "solicitud": {"producto_id": "PEC-01-0001", "cantidad": 2, "presupuesto_centavos": 100000, "plazo_max_dias": 7}, "faltantes": []}']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	PromptInput([PromptInput]):::first
	ChatPromptTemplate(ChatProm

LCEL expone etapas declarativas. @chain envuelve la misma extracción. Los faltantes se recalculan tras el parser.

## R01 · anatomía del agente
La anatomía se lee de los objetos reales del agente final (`pedro.anatomia`), no se escribe a mano. No se llama a ningún modelo: el executor se construye con un modelo de relleno que nunca se invoca.

In [3]:
import json
from pedro.catalog import crear_escenario
from pedro.anatomia import describir_anatomia
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
memoria = Path('storage/notebooks/r01.sqlite3')
anatomia = describir_anatomia(escenario, memoria)
for clave in ('motor_cognitivo', 'orquestador', 'system_message', 'memoria',
              'base_conocimiento', 'puerta_publicacion'):
    print(clave, json.dumps(anatomia[clave], ensure_ascii=False))
for h in anatomia['herramientas']:
    print('herramienta', h['nombre'], '|', h['papel'], '|', h['logica'], '|', h['argumentos'])

motor_cognitivo {"clase": "ChatOpenAI", "modelo": "gpt-4.1-mini", "disponible": false}
orquestador {"agente": "create_openai_tools_agent", "ejecutor": "AgentExecutor", "max_iteraciones": 6, "historial": "RunnableWithMessageHistory"}
system_message {"ptcf": {"Persona": true, "Tarea": true, "Contexto": true, "Formato": true}, "variables": ["agent_scratchpad", "input"], "variables_fijadas": ["chat_history", "dataset_id", "ejemplos", "formato"], "mensajes": ["SystemMessagePromptTemplate", "MessagesPlaceholder", "HumanMessagePromptTemplate", "MessagesPlaceholder"]}
memoria {"corto_plazo": "ventana de 12 mensajes + borrador por sesión", "largo_plazo": "preferencias explícitas + episodios vectoriales por usuario", "modelo_embeddings": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
base_conocimiento {"dataset_id": "7012faf4-4831-4b29-b230-8c4ad32075a8", "piezas": 2640, "fragmentos_politicas": 75, "origen_politicas": "ficticio_demo"}
puerta_publicacion {"verificacion": "exacta (

## R01 · agente híbrido: capa reactiva y capa deliberativa
Tres turnos sobre el mismo flujo final. **Doble de prueba:** el modelo es un guion determinista (no es OpenAI) y ViaCEP está simulado. Lo real es el flujo: las reglas, el `AgentExecutor`, la herramienta ejecutada y la verificación antes de publicar. Los dos primeros turnos los resuelve una regla sin llamar al modelo; el tercero pasa por el modelo, que recibe las seis herramientas y solicita una.

In [4]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Traza, crear_flujo
from pedro.memory import nueva_sesion
sesion = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
final = json.dumps({'estado': 'cep', 'dataset_id': escenario.dataset_id,
    'solicitud': {'cep': '01001000'}, 'cidade': 'Cidade Simulada', 'uf': 'SP',
    'modo_cep': 'simulado', 'evidencias': [{'tipo': 'viacep',
    'fuente_id': 'ViaCEP:01001000', 'texto': 'Cidade Simulada/SP'}]})
guion = [AIMessage(content='', tool_calls=[{'name': 'consultar_cep',
                                             'args': {'cep': '01001000'}, 'id': 'c1'}]),
         AIMessage(content=final)]
llamadas = []
def modelo_doble(prompt, **kwargs):
    llamadas.append(sorted(t['function']['name'] for t in kwargs.get('tools', [])))
    return guion[len(llamadas) - 1]
traza = Traza()
flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),
                    cep_simulado=True, traza=traza)
config = {'configurable': {'user_id': 'alumna', 'session_id': sesion}}
for pregunta in ('lista piezas', 'Compara PEC-01-0001',
                 '¿A qué ciudad corresponde el CEP 01001000?'):
    antes = (len(llamadas), len(traza.eventos))
    respuesta = flujo.invoke({'input': pregunta}, config=config).content
    usadas = [e['herramienta'] for e in traza.eventos[antes[1]:] if e['evento'] == 'tool_start']
    n = len(llamadas) - antes[0]
    print('turno:', pregunta)
    print('  capa:', 'deliberativa' if n else 'reactiva', '| llamadas al modelo:', n,
          '| herramientas ejecutadas:', usadas)
    print('  respuesta:', respuesta[:100].replace('\n', ' '))
print('herramientas ofrecidas al modelo:', llamadas[0])
assert len(llamadas) == 2 and [e['herramienta'] for e in traza.eventos
                               if e['evento'] == 'tool_start'] == ['consultar_cep']

turno: lista piezas
  capa: reactiva | llamadas al modelo: 0 | herramientas ejecutadas: []
  respuesta: Piezas del escenario (2640; página 1): - PEC-01-0001 · NEVE-COMP-0001 · Compresor para Neveras NEVE-
turno: Compara PEC-01-0001
  capa: reactiva | llamadas al modelo: 0 | herramientas ejecutadas: []
  respuesta: ¿Cuántas unidades de PEC-01-0001 necesitas? Puedo comparar sin presupuesto o plazo máximo si no tien
turno: ¿A qué ciudad corresponde el CEP 01001000?
  capa: deliberativa | llamadas al modelo: 2 | herramientas ejecutadas: ['consultar_cep']
  respuesta: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
herramientas ofrecidas al modelo: ['buscar_productos', 'comparar_ofertas', 'consultar_catalogo', 'consultar_cep', 'consultar_politicas', 'guardar_preferencia']


/tmp/ipykernel_3061218/4119760277.py:18: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),


Lectura: «lista piezas» y «Compara PEC-01-0001» se resuelven por regla (0 llamadas al modelo). La pregunta de CEP no tiene regla: el modelo elige `consultar_cep` entre seis herramientas y la respuesta solo se publica tras coincidir con la observación de la herramienta. Con un modelo real la elección la hace el LLM; esa ejecución remota corresponde a R26.

## R02 · planificación: una solicitud compleja dividida en pasos
Ejecución real local, sin modelo. La solicitud se divide en cinco subtareas, cada una con una salida observable: extraer restricciones, consultar ofertas, comparar, verificar y publicar.

In [5]:
from pedro.agent import renderizar
from pedro.business import comparar
from pedro.catalog import ofertas
from pedro.dialogue import interpretar_compra
from pedro.verification import respuesta_local, verificar
mensaje = 'Necesito 2 unidades de PEC-01-0001 hasta R$ 5.000 en 7 días'
estado, aclaracion = interpretar_compra(escenario, None, mensaje)
print('1 extraer  ', estado.solicitud.model_dump(exclude_none=True), '| falta:', aclaracion)
disponibles = ofertas(escenario, estado.solicitud.producto_id)
print('2 consultar', [o.id for o in disponibles])
alternativas = comparar(estado.solicitud, disponibles)
print('3 comparar ', [(a.oferta.proveedor_id, a.total_centavos, a.cumple, a.motivos)
                      for a in alternativas])
propuesta = respuesta_local(escenario, estado.solicitud)
print('4 verificar', 'errores:', verificar(propuesta, escenario))
print('5 publicar ', renderizar(propuesta).splitlines()[0])

1 extraer   {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'presupuesto_centavos': 500000, 'plazo_max_dias': 7} | falta: None
2 consultar ['PEC-01-0001-BR-05', 'PEC-01-0001-BR-06', 'PEC-01-0001-INT-06']
3 comparar  [('BR-06', 296307, True, []), ('BR-05', 81980, False, ['stock', 'plazo']), ('INT-06', 320027, False, ['plazo'])]
4 verificar errores: []
5 publicar  CentroPeças: R$ 2.963,07 para 2 unidades; plazo 2 días.


## R02 · memoria aumentada: qué va a corto y a largo plazo
Ejecución real local, sin modelo. Corto plazo: el borrador de compra y la ventana de 12 mensajes viven en la sesión. Largo plazo: la preferencia vive en el usuario y aparece en una sesión nueva; otro usuario no la ve. Los episodios vectoriales se demuestran en el notebook 03.

In [6]:
from pedro.memory import Historial, guardar_preferencia, leer_borrador, preferencias
sesion_a = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
local = crear_flujo(escenario, memoria)
def cfg(usuario, sesion):
    return {'configurable': {'user_id': usuario, 'session_id': sesion}}
for texto in ('Necesito 2 unidades de PEC-01-0001 en 7 días', 'ahora 3 unidades'):
    salida = local.invoke({'input': texto}, config=cfg('alumna', sesion_a)).content
    print(texto, '->', salida.splitlines()[0])
    print('  borrador (corto plazo):', {k: v for k, v in
          leer_borrador(memoria, 'alumna', sesion_a)['solicitud'].items() if v is not None})
for _ in range(7):
    local.invoke({'input': 'lista piezas'}, config=cfg('alumna', sesion_a))
historial = Historial(memoria, 'alumna', sesion_a)
print('mensajes guardados:', len(historial.todos()), '| ventana que ve el agente:', len(historial.messages))
guardar_preferencia(memoria, 'alumna', 'proveedor', 'nacional', explicita=True)
sesion_b = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
print('sesión nueva -> borrador:', leer_borrador(memoria, 'alumna', sesion_b),
      '| preferencia (largo plazo):', preferencias(memoria, 'alumna'),
      '| otro usuario:', preferencias(memoria, 'otro'))

/tmp/ipykernel_3061218/307983916.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  local = crear_flujo(escenario, memoria)


Necesito 2 unidades de PEC-01-0001 en 7 días -> CentroPeças: R$ 2.963,07 para 2 unidades; plazo 2 días.
  borrador (corto plazo): {'producto_id': 'PEC-01-0001', 'cantidad': 2, 'plazo_max_dias': 7}
ahora 3 unidades -> CentroPeças: R$ 4.401,99 para 3 unidades; plazo 2 días.
  borrador (corto plazo): {'producto_id': 'PEC-01-0001', 'cantidad': 3, 'plazo_max_dias': 7}


mensajes guardados: 18 | ventana que ve el agente: 12


sesión nueva -> borrador: None | preferencia (largo plazo): {'proveedor': 'nacional'} | otro usuario: {}


Lectura: al cambiar la cantidad a 3 el plazo de 7 días se conserva en el borrador. El historial guarda 18 mensajes pero el agente solo ve 12. La sesión nueva empieza sin borrador y conserva la preferencia del usuario.